In [ ]:
# Cell 1: Import Libraries
# ---------------------
import pandas as pd
import numpy as np
import yfinance as yf

import matplotlib.pyplot as plt
from datetime import datetime, timedelta

In [ ]:
# Cell 2: Import problematic Libraries
# ---------------------
# Install pandas_ta but don't import it yet
try:
    import pandas_ta as ta
except ImportError:
    !pip install pandas_ta -q

    # Fix the compatibility issue immediately after installation
    import glob
    pandas_ta_paths = glob.glob("/usr/local/lib/python3.*/dist-packages/pandas_ta/momentum/squeeze_pro.py")

    if pandas_ta_paths:
        for path in pandas_ta_paths:
            !sed -i 's/from numpy import NaN as npNaN/from numpy import nan as npNaN/' {path}
            print(f"Fixed pandas_ta compatibility issue at: {path}")
    else:
        print("pandas_ta path not found automatically. Manual fix may be required.")

    # Now try importing
    import pandas_ta as ta

In [ ]:
#cell 3  Candlestick Pattern Detection Helper Functions
# --------------------------------------------
# These functions detect common candlestick patterns in price data
# and return True when patterns are detected, False otherwise.

def detect_hammer(df, body_ratio=0.3, shadow_ratio=2.0):
    """
    Detect hammer candlestick patterns

    A hammer has:
    - Small body at the upper portion of the range
    - Long lower shadow (at least 2x the body)
    - Little or no upper shadow

    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame with OHLCV data
    body_ratio : float
        Maximum ratio of body to total range (default 0.3 = 30%)
    shadow_ratio : float
        Minimum ratio of lower shadow to body (default 2.0 = 200%)

    Returns:
    --------
    pd.Series
        Series with Boolean values: False (no pattern) or True (hammer pattern)
    """
    # Calculate parts of the candle
    body_size = abs(df['Close'] - df['Open'])
    total_range = df['High'] - df['Low']

    # Determine the body position (top or bottom)
    body_high = df[['Open', 'Close']].max(axis=1)
    body_low = df[['Open', 'Close']].min(axis=1)

    # Calculate shadows
    upper_shadow = df['High'] - body_high
    lower_shadow = body_low - df['Low']

    # Avoid division by zero
    body_size_safe = body_size.copy()
    body_size_safe = body_size_safe.replace(0, 0.0001)
    total_range_safe = total_range.copy()
    total_range_safe = total_range_safe.replace(0, 0.0001)

    # Define hammer criteria
    is_hammer = (
        # 1. Small body relative to total range
        (body_size <= body_ratio * total_range_safe) &

        # 2. Long lower shadow relative to body
        (lower_shadow >= shadow_ratio * body_size_safe) &

        # 3. Small or no upper shadow
        (upper_shadow <= 0.1 * total_range_safe) &

        # 4. Body in upper portion of the range
        (body_low >= (df['Low'] + 0.6 * total_range_safe))
    )

    # Return Boolean values
    return is_hammer.astype(bool)

def detect_shooting_star(df, body_ratio=0.3, shadow_ratio=2.0):
    """
    Detect shooting star candlestick patterns

    A shooting star has:
    - Small body at the lower portion of the range
    - Long upper shadow (at least 2x the body)
    - Little or no lower shadow

    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame with OHLCV data
    body_ratio : float
        Maximum ratio of body to total range (default 0.3 = 30%)
    shadow_ratio : float
        Minimum ratio of upper shadow to body (default 2.0 = 200%)

    Returns:
    --------
    pd.Series
        Series with Boolean values: False (no pattern) or True (shooting star pattern)
    """
    # Calculate parts of the candle
    body_size = abs(df['Close'] - df['Open'])
    total_range = df['High'] - df['Low']

    # Determine the body position
    body_high = df[['Open', 'Close']].max(axis=1)
    body_low = df[['Open', 'Close']].min(axis=1)

    # Calculate shadows
    upper_shadow = df['High'] - body_high
    lower_shadow = body_low - df['Low']

    # Avoid division by zero
    body_size_safe = body_size.copy()
    body_size_safe = body_size_safe.replace(0, 0.0001)
    total_range_safe = total_range.copy()
    total_range_safe = total_range_safe.replace(0, 0.0001)

    # Define shooting star criteria
    is_shooting_star = (
        # 1. Small body relative to total range
        (body_size <= body_ratio * total_range_safe) &

        # 2. Long upper shadow relative to body
        (upper_shadow >= shadow_ratio * body_size_safe) &

        # 3. Small or no lower shadow
        (lower_shadow <= 0.1 * total_range_safe) &

        # 4. Body in lower portion of the range
        (body_high <= (df['Low'] + 0.4 * total_range_safe))
    )

    # Return Boolean values
    return is_shooting_star.astype(bool)

def detect_doji(df, doji_ratio=0.05):
    """
    Detect doji candlestick patterns

    A doji has:
    - Almost no body (open and close are very close)
    - Upper and lower shadows can vary

    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame with OHLCV data
    doji_ratio : float
        Maximum ratio of body to total range to be considered a doji (default 0.05 = 5%)

    Returns:
    --------
    pd.Series
        Series with Boolean values: False (no pattern) or True (doji pattern)
    """
    # Calculate body size and total range
    body_size = abs(df['Close'] - df['Open'])
    total_range = df['High'] - df['Low']

    # Avoid division by zero
    total_range_safe = total_range.copy()
    total_range_safe = total_range_safe.replace(0, 0.0001)

    # A doji has a very small body compared to its range
    is_doji = body_size <= (doji_ratio * total_range_safe)

    # Return Boolean values
    return is_doji.astype(bool)

def detect_shooting_star(df, body_ratio=0.3, shadow_ratio=2.0):
    """
    Detect shooting star candlestick patterns

    A shooting star has:
    - Small body at the lower portion of the range
    - Long upper shadow (at least 2x the body)
    - Little or no lower shadow

    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame with OHLCV data
    body_ratio : float
        Maximum ratio of body to total range (default 0.3 = 30%)
    shadow_ratio : float
        Minimum ratio of upper shadow to body (default 2.0 = 200%)

    Returns:
    --------
    pd.Series
        Series with Boolean values: False (no pattern) or True (shooting star pattern)
    """
    # Calculate parts of the candle
    body_size = abs(df['Close'] - df['Open'])
    total_range = df['High'] - df['Low']

    # Determine the body position
    body_high = df[['Open', 'Close']].max(axis=1)
    body_low = df[['Open', 'Close']].min(axis=1)

    # Calculate shadows
    upper_shadow = df['High'] - body_high
    lower_shadow = body_low - df['Low']

    # Avoid division by zero
    body_size_safe = body_size.copy()
    body_size_safe = body_size_safe.replace(0, 0.0001)
    total_range_safe = total_range.copy()
    total_range_safe = total_range_safe.replace(0, 0.0001)

    # Define shooting star criteria
    is_shooting_star = (
        # 1. Small body relative to total range
        (body_size <= body_ratio * total_range_safe) &

        # 2. Long upper shadow relative to body
        (upper_shadow >= shadow_ratio * body_size_safe) &

        # 3. Small or no lower shadow
        (lower_shadow <= 0.1 * total_range_safe) &

        # 4. Body in lower portion of the range - more flexible position requirement
        (body_high <= (df['Low'] + 0.5 * total_range_safe))
    )

    # Return Boolean values
    return is_shooting_star.astype(bool)
# Use this to fix your bearish engulfing detection:
def detect_engulfing(df):
    """
     detect_engulfing
    """
    # For previous candle
    prev_open = df['Open'].shift(1)
    prev_close = df['Close'].shift(1)
    prev_high = df['High'].shift(1)
    prev_low = df['Low'].shift(1)

    # For current candle
    curr_open = df['Open']
    curr_close = df['Close']
    curr_high = df['High']
    curr_low = df['Low']

    # Determine candle directions
    prev_bullish = prev_close > prev_open
    prev_bearish = prev_close < prev_open
    curr_bullish = curr_close > curr_open
    curr_bearish = curr_close < curr_open

    # Bullish engulfing: previous bearish, current bullish, current body engulfs previous
    bullish_engulfing = (
        prev_bearish &          # Previous candle was bearish
        curr_bullish &          # Current candle is bullish
        (curr_open < prev_close) &  # Current open below previous close
        (curr_close > prev_open)    # Current close above previous open
    )

    # Bearish engulfing: previous bullish, current bearish, current body engulfs previous
    bearish_engulfing = (
        prev_bullish &          # Previous candle was bullish
        curr_bearish &          # Current candle is bearish
        (curr_open > prev_close) &  # Current open above previous close
        (curr_close < prev_open)    # Current close below previous open
    )

    # Count the number of patterns detected
    total_bearish = bearish_engulfing.sum()
    total_bullish = bullish_engulfing.sum()
    total_rows = len(df)

    print(f"Total rows: {total_rows}")
    print(f"Bullish engulfing patterns: {total_bullish} ({total_bullish/total_rows*100:.2f}%)")
    print(f"Bearish engulfing patterns: {total_bearish} ({total_bearish/total_rows*100:.2f}%)")

    return (bullish_engulfing, bearish_engulfing)
# Example of how to use these functions in the add_indicators function:
# def add_indicators(df):
#     # ... existing code for other indicators
#
#     # Add candlestick patterns
#     data['HAMMER'] = detect_hammer(data)
#     data['SHOOTING_STAR'] = detect_shooting_star(data)
#     data['DOJI'] = detect_doji(data)
#
#     bullish_engulfing, bearish_engulfing = detect_engulfing(data)
#     data['BULLISH_ENGULFING'] = bullish_engulfing
#     data['BEARISH_ENGULFING'] = bearish_engulfing
#
#     return data

In [ ]:
# Cell 4: Backtesting Engine
# ------------------------
class PriceSeriesWrapper:
    """
    Wrapper for price series to allow for EasyLanguage-style indexing
    """
    def __init__(self, series, current_idx):
        self.series = series
        self.current_idx = current_idx

    def __getitem__(self, offset):
        """
        Get the value at the specified offset

        Parameters:
        -----------
        offset : int
            Offset from current position (0 = current, 1 = previous, etc.)

        Returns:
        --------
        float
            The value at the specified offset
        """
        idx = self.current_idx - offset
        if idx < 0:
            raise IndexError(f"Offset {offset} exceeds available data")
        return self.series.iloc[idx]

    def lowest(self, periods):
        """
        Get the lowest value over the specified number of periods

        Parameters:
        -----------
        periods : int
            Number of periods to look back

        Returns:
        --------
        float
            The lowest value over the specified periods
        """
        start_idx = max(0, self.current_idx - periods + 1)
        return self.series.iloc[start_idx:self.current_idx + 1].min()

    def highest(self, periods):
        """
        Get the highest value over the specified number of periods

        Parameters:
        -----------
        periods : int
            Number of periods to look back

        Returns:
        --------
        float
            The highest value over the specified periods
        """
        start_idx = max(0, self.current_idx - periods + 1)
        return self.series.iloc[start_idx:self.current_idx + 1].max()

class BacktestEngine:
    """
    Simple backtesting engine for trading strategies
    """
    def __init__(self, data, ticker, initial_capital=10000):
        """
        Initialize the backtesting engine

        Parameters:
        -----------
        data : pd.DataFrame
            DataFrame with price data and indicators
        ticker : str
            Ticker symbol being traded
        initial_capital : float
            Initial capital in USD
        """
        self.data = data.copy()
        self.ticker = ticker
        self.initial_capital = initial_capital

        # Prepare results storage
        self.positions = []
        self.trades = []
        self.equity = []

        # Current state
        self.current_position = 0  # 0=no position, 1=long, -1=short
        self.entry_price = 0
        self.entry_date = None
        self.current_capital = initial_capital
        self.current_shares = 0
        self.bars_since_entry = 0

        print(f"Backtesting engine initialized for {ticker} with ${initial_capital:,.2f} capital")

    def run_backtest(self, strategy_function, trade_amount=None, stop_loss_usd=None, debug=False):
        """
        Run the backtest using the provided strategy function

        Parameters:
        -----------
        strategy_function : function
            A function that takes a dictionary of price/indicator wrappers and returns a dict with signals
        trade_amount : float or None
            USD amount to use for each trade (if None, uses all available capital)
        stop_loss_usd : float or None
            Stop loss amount in USD (if None, no stop loss is applied)
        debug : bool
            Whether to print debug information during the backtest

        Returns:
        --------
        dict
            Dictionary with backtest results
        """
        if trade_amount is None:
            trade_amount = self.initial_capital

        print(f"Running backtest with ${trade_amount:,.2f} per trade...")
        if stop_loss_usd is not None:
            print(f"Stop loss set at ${stop_loss_usd:,.2f} per trade")

        # Reset state
        self.positions = []
        self.trades = []
        self.equity = []
        self.current_position = 0
        self.entry_price = 0
        self.entry_date = None
        self.current_capital = self.initial_capital
        self.current_shares = 0
        self.entry_value = 0  # Track the USD value at entry for stop loss calculation
        self.current_equity = self.initial_capital  # Track current equity for accurate calculations
        self.trade_capital = trade_amount  # Store the fixed trade amount
        self.bars_since_entry = 0
        # Initialize equity curve with starting capital
        self.equity.append({
            'date': self.data.index[0],
            'equity': self.initial_capital,
            'position': 0
        })

        # Loop through each bar (starting from index 1 to avoid look-ahead bias)
        for i in range(1, len(self.data) - 1):
            current_date = self.data.index[i]
            current_row = self.data.iloc[i]
            next_row = self.data.iloc[i+1]  # For execution on next bar's open
            # Add this code to update bars_since_entry:
            if self.current_position != 0:

                # If we're in a position
                self.bars_since_entry += 1
            else:
              self.bars_since_entry = 0

            # Get current prices
            current_close = current_row['Close']
            next_open = next_row['Open']

            # Create wrappers for each series in the dataframe
            wrappers = {}
            for column in self.data.columns:
                wrappers[column] = PriceSeriesWrapper(self.data[column], i)

            # Convenience variables for common price data
            wrappers['open'] = wrappers['Open']
            wrappers['high'] = wrappers['High']
            wrappers['low'] = wrappers['Low']
            wrappers['close'] = wrappers['Close']
            wrappers['volume'] = wrappers['Volume']
            wrappers['bars_since_entry'] = self.bars_since_entry

            # Check for stop loss if we have a position
            stop_loss_triggered = False
            if stop_loss_usd is not None and self.current_position != 0:
                if self.current_position == 1:  # Long position
                    # Calculate current value and check if loss exceeds stop loss
                    current_value = self.current_shares * current_close
                    current_pnl = current_value - self.entry_value
                    if current_pnl <= -stop_loss_usd:
                        stop_loss_triggered = True
                elif self.current_position == -1:  # Short position
                    # For short position, loss is when price goes up
                    current_pnl = self.entry_value - (self.current_shares * current_close)
                    if current_pnl <= -stop_loss_usd:
                        stop_loss_triggered = True

            # Get strategy signals
            signals = strategy_function(wrappers)

            if debug and (signals or stop_loss_triggered or self.current_position != 0):
                print(f"\nDate: {current_date}")
                print(f"  Position: {self.current_position} ({self.current_shares:.2f} shares at ${self.entry_price:.2f})")
                print(f"  Signals: {signals}")
                print(f"  Stop Loss Triggered: {stop_loss_triggered}")
                print(f"  Current Close: ${current_close:.2f}")
                print(f"  Next Open: ${next_open:.2f}")
                print(f"  Current Capital: ${self.current_capital:.2f}")
                print(f"  Current Equity: ${self.current_equity:.2f}")

            # Handle existing position exit
            if self.current_position != 0:
                if (self.current_position == 1 and (signals.get('exit_long', False) or stop_loss_triggered)) or \
                   (self.current_position == -1 and (signals.get('exit_short', False) or stop_loss_triggered)):

                    # Calculate P&L
                    exit_price = next_open  # Execute on next bar's open
                    exit_date = self.data.index[i+1]

                    if self.current_position == 1:  # Long position
                        pnl = (exit_price - self.entry_price) * self.current_shares
                        # Update capital (add the current value of shares)
                        self.current_capital += self.current_shares * exit_price
                    else:  # Short position
                        pnl = (self.entry_price - exit_price) * self.current_shares
                        # For short position: return borrowed shares and update capital
                        self.current_capital += self.entry_value + pnl

                    # Record the trade
                    self.trades.append({
                        'entry_date': self.entry_date,
                        'exit_date': exit_date,
                        'position': 'Long' if self.current_position == 1 else 'Short',
                        'entry_price': self.entry_price,
                        'exit_price': exit_price,
                        'shares': self.current_shares,
                        'pnl': pnl,
                        'pnl_pct': pnl / (self.entry_price * self.current_shares) * 100,
                        'stop_loss': stop_loss_triggered,
                        'exit_signal': 'stop_loss' if stop_loss_triggered else ('exit_long' if self.current_position == 1 else 'exit_short')
                    })

                    if debug:
                        print(f"  Trade Closed:")
                        print(f"    Entry: {self.entry_date} at ${self.entry_price:.2f}")
                        print(f"    Exit: {exit_date} at ${exit_price:.2f}")
                        print(f"    P&L: ${pnl:.2f} ({pnl / (self.entry_price * self.current_shares) * 100:.2f}%)")
                        print(f"    Reason: {'Stop Loss' if stop_loss_triggered else 'Exit Signal'}")

                    # Reset position
                    self.current_position = 0
                    self.current_shares = 0
                    self.entry_price = 0
                    self.entry_date = None
                    self.bars_since_entry = 0

            # Handle new position entry (only if we don't have a position)
            if self.current_position == 0:
                if signals.get('enter_long', False):
                    # Calculate number of shares to buy
                    self.entry_price = next_open  # Execute on next bar's open
                    self.entry_date = self.data.index[i+1]

                    # Use either trade_amount or all available capital
                    amount_to_use = min(trade_amount, self.current_capital)
                    self.current_shares = amount_to_use / self.entry_price

                    # Update capital and position
                    self.current_capital -= self.current_shares * self.entry_price
                    self.entry_value = self.current_shares * self.entry_price
                    self.current_position = 1

                    if debug:
                        print(f"  Long Entry:")
                        print(f"    Entry Date: {self.entry_date}")
                        print(f"    Entry Price: ${self.entry_price:.2f}")
                        print(f"    Shares: {self.current_shares:.2f}")
                        print(f"    Position Value: ${self.entry_value:.2f}")
                        print(f"    Remaining Capital: ${self.current_capital:.2f}")

                elif signals.get('enter_short', False):
                    # Calculate number of shares to short
                    self.entry_price = next_open  # Execute on next bar's open
                    self.entry_date = self.data.index[i+1]

                    # Use either trade_amount or all available capital
                    amount_to_use = min(trade_amount, self.current_capital)
                    self.current_shares = amount_to_use / self.entry_price

                    # Short selling: borrow shares and sell them
                    self.current_capital += self.current_shares * self.entry_price
                    self.entry_value = self.current_shares * self.entry_price
                    self.current_position = -1

                    if debug:
                        print(f"  Short Entry:")
                        print(f"    Entry Date: {self.entry_date}")
                        print(f"    Entry Price: ${self.entry_price:.2f}")
                        print(f"    Shares: {self.current_shares:.2f}")
                        print(f"    Position Value: ${self.entry_value:.2f}")
                        print(f"    Remaining Capital: ${self.current_capital:.2f}")

            # Record position
            self.positions.append({
                'date': current_date,
                'position': self.current_position,
                'shares': self.current_shares,
                'entry_price': self.entry_price if self.current_position != 0 else 0
            })

            # Calculate current equity
            # This should be: initial capital + sum of all trade P&Ls (fixed trade amount approach)
            # NOT: cash + current position value (which creates compounding effect)
            total_trade_pnl = sum(trade['pnl'] for trade in self.trades)
            self.current_equity = self.initial_capital + total_trade_pnl

            # Update equity curve
            self.equity.append({
                'date': current_date,
                'equity': self.current_equity,
                'position': self.current_position
            })

        # Convert results to DataFrames
        self.equity_curve = pd.DataFrame(self.equity).set_index('date')
        self.trade_history = pd.DataFrame(self.trades)

        # Calculate performance metrics
        self.results = self._calculate_performance_metrics()

        return self.results

    def _calculate_performance_metrics(self):
        """
        Calculate performance metrics from backtest results

        Returns:
        --------
        dict
            Dictionary with performance metrics
        """
        # Prepare results dictionary
        results = {}

        # Exit if no trades were made
        if len(self.trades) == 0:
            print("No trades were executed during the backtest.")
            return {
                'total_trades': 0,
                'net_profit': 0,
                'profit_factor': 0,
                'win_rate': 0,
                'max_drawdown': 0,
                'sharpe_ratio': 0,
                'sortino_ratio': 0
            }

        # Basic trade metrics
        results['total_trades'] = len(self.trades)
        results['net_profit'] = sum(trade['pnl'] for trade in self.trades)

        # Final equity vs initial capital for percentage calculation
        final_equity = self.equity_curve['equity'].iloc[-1]
        results['net_profit_pct'] = (final_equity / self.initial_capital - 1) * 100

        # Win/Loss metrics
        winning_trades = [trade for trade in self.trades if trade['pnl'] > 0]
        losing_trades = [trade for trade in self.trades if trade['pnl'] <= 0]

        results['winning_trades'] = len(winning_trades)
        results['losing_trades'] = len(losing_trades)
        results['win_rate'] = len(winning_trades) / len(self.trades) if len(self.trades) > 0 else 0

        # Profit metrics
        total_profit = sum(trade['pnl'] for trade in winning_trades) if winning_trades else 0
        total_loss = sum(trade['pnl'] for trade in losing_trades) if losing_trades else 0

        results['gross_profit'] = total_profit
        results['gross_loss'] = total_loss
        results['profit_factor'] = abs(total_profit / total_loss) if total_loss != 0 else float('inf')

        # Average trade metrics
        results['avg_trade'] = results['net_profit'] / len(self.trades) if len(self.trades) > 0 else 0
        results['avg_winning_trade'] = total_profit / len(winning_trades) if len(winning_trades) > 0 else 0
        results['avg_losing_trade'] = total_loss / len(losing_trades) if len(losing_trades) > 0 else 0

        # Streak metrics
        current_streak = 0
        max_winning_streak = 0
        max_losing_streak = 0

        for trade in self.trades:
            if trade['pnl'] > 0:
                if current_streak > 0:
                    current_streak += 1
                else:
                    current_streak = 1
            else:
                if current_streak < 0:
                    current_streak -= 1
                else:
                    current_streak = -1

            max_winning_streak = max(max_winning_streak, current_streak if current_streak > 0 else 0)
            max_losing_streak = min(max_losing_streak, current_streak if current_streak < 0 else 0)

        results['max_winning_streak'] = max_winning_streak
        results['max_losing_streak'] = abs(max_losing_streak)

        # Calculate drawdown
        equity_series = self.equity_curve['equity']
        rolling_max = equity_series.cummax()
        drawdown = (equity_series - rolling_max) / rolling_max * 100
        results['max_drawdown'] = abs(drawdown.min())

        # Calculate Sharpe and Sortino ratios
        daily_returns = equity_series.pct_change().dropna()

        # Annualized Sharpe Ratio (assuming 252 trading days in a year)
        risk_free_rate = 0.02 / 252  # 2% annual risk-free rate
        excess_returns = daily_returns - risk_free_rate

        if len(daily_returns) > 0:
            sharpe_ratio = np.sqrt(252) * excess_returns.mean() / daily_returns.std()

            # Sortino ratio (only considering negative returns)
            negative_returns = daily_returns[daily_returns < 0]
            sortino_ratio = np.sqrt(252) * excess_returns.mean() / negative_returns.std() if len(negative_returns) > 0 else np.inf

            results['sharpe_ratio'] = sharpe_ratio
            results['sortino_ratio'] = sortino_ratio
        else:
            results['sharpe_ratio'] = 0
            results['sortino_ratio'] = 0

        return results

    def _get_contiguous_periods(self, mask):
        """
        Get a list of start and end dates for contiguous periods
        where the mask is True

        Parameters:
        -----------
        mask : pd.Series
            Boolean mask indicating where a condition is True

        Returns:
        --------
        list
            List of (start_date, end_date) tuples
        """
        periods = []
        if not mask.any():
            return periods

        # Convert to numpy array for processing
        mask_array = mask.values
        dates = mask.index

        # Find the start and end indices of True segments
        diff = np.diff(np.concatenate(([False], mask_array, [False])))
        starts = np.where(diff == 1)[0]
        ends = np.where(diff == -1)[0] - 1

        # Fix for empty arrays
        if len(starts) == 0 or len(ends) == 0:
            return periods

        # Create list of (start_date, end_date) tuples
        for i in range(len(starts)):
            periods.append((dates[starts[i]], dates[ends[i]]))

        return periods

    def plot_results(self):
        """
        Plot the backtest results
        """
        if len(self.equity) == 0:
            print("No backtest results to plot. Run the backtest first.")
            return

        # Create a figure with two subplots (equity curve and drawdown)
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), gridspec_kw={'height_ratios': [3, 1]})

        # Plot equity curve
        self.equity_curve['equity'].plot(ax=ax1, color='blue', linewidth=2)
        ax1.set_title(f'Backtest Results: {self.ticker}', fontsize=16)
        ax1.set_ylabel('Portfolio Value ($)', fontsize=12)
        ax1.grid(True)

        # Create position masks for highlighting
        long_mask = self.equity_curve['position'] == 1
        short_mask = self.equity_curve['position'] == -1

        # Highlight position periods
        if long_mask.any():
            long_periods = self._get_contiguous_periods(long_mask)
            for start, end in long_periods:
                ax1.axvspan(start, end, alpha=0.2, color='green')

        if short_mask.any():
            short_periods = self._get_contiguous_periods(short_mask)
            for start, end in short_periods:
                ax1.axvspan(start, end, alpha=0.2, color='red')

        # Calculate and plot drawdown
        equity_series = self.equity_curve['equity']
        rolling_max = equity_series.cummax()
        drawdown = (equity_series - rolling_max) / rolling_max * 100

        drawdown.plot(ax=ax2, color='red', linewidth=2)
        ax2.set_title('Drawdown (%)', fontsize=14)
        ax2.set_ylabel('Drawdown (%)', fontsize=12)
        ax2.set_xlabel('Date', fontsize=12)
        ax2.grid(True)

        # Adjust layout and display plot
        plt.tight_layout()
        plt.show()

        # Display summary table
        print("\nPerformance Summary:")
        for key, value in self.results.items():
            if isinstance(value, float):
                if key in ['net_profit', 'gross_profit', 'gross_loss', 'avg_trade', 'avg_winning_trade', 'avg_losing_trade']:
                    print(f"{key.replace('_', ' ').title()}: ${value:,.2f}")
                elif key in ['net_profit_pct', 'max_drawdown']:
                    print(f"{key.replace('_', ' ').title()}: {value:.2f}%")
                elif key == 'win_rate':
                    print(f"{key.replace('_', ' ').title()}: {value*100:.2f}%")
                else:
                    print(f"{key.replace('_', ' ').title()}: {value:.4f}")
            else:
                print(f"{key.replace('_', ' ').title()}: {value}")

    def analyze_trade_statistics(self):
        """
        Analyze trade statistics beyond the basic performance metrics

        Returns:
        --------
        dict
            Dictionary with additional trade statistics
        """
        if not hasattr(self, 'trade_history') or self.trade_history.empty:
            print("No trades to analyze. Run the backtest first.")
            return {}

        stats = {}

        # Convert trade history to DataFrame if it's a list
        trades_df = self.trade_history.copy() if isinstance(self.trade_history, pd.DataFrame) else pd.DataFrame(self.trades)

        # Add trade duration
        if 'entry_date' in trades_df.columns and 'exit_date' in trades_df.columns:
            trades_df['duration'] = (pd.to_datetime(trades_df['exit_date']) -
                                   pd.to_datetime(trades_df['entry_date'])).dt.days

            stats['avg_trade_duration'] = trades_df['duration'].mean()
            stats['avg_winning_duration'] = trades_df.loc[trades_df['pnl'] > 0, 'duration'].mean()
            stats['avg_losing_duration'] = trades_df.loc[trades_df['pnl'] <= 0, 'duration'].mean()

        # Analyze stop losses
        if 'stop_loss' in trades_df.columns:
            stop_loss_trades = trades_df[trades_df['stop_loss'] == True]
            stats['stop_loss_trades'] = int(len(stop_loss_trades))
            stats['stop_loss_percentage'] = len(stop_loss_trades) / len(trades_df) * 100 if len(trades_df) > 0 else 0
            stats['stop_loss_avg_loss'] = stop_loss_trades['pnl'].mean() if len(stop_loss_trades) > 0 else 0

        # Analyze by position type
        if 'position' in trades_df.columns:
            long_trades = trades_df[trades_df['position'] == 'Long']
            short_trades = trades_df[trades_df['position'] == 'Short']

            stats['long_trades'] = int(len(long_trades))
            stats['short_trades'] = int(len(short_trades))

            stats['long_win_rate'] = len(long_trades[long_trades['pnl'] > 0]) / len(long_trades) * 100 if len(long_trades) > 0 else 0
            stats['short_win_rate'] = len(short_trades[short_trades['pnl'] > 0]) / len(short_trades) * 100 if len(short_trades) > 0 else 0

            stats['long_avg_profit'] = long_trades['pnl'].mean() if len(long_trades) > 0 else 0
            stats['short_avg_profit'] = short_trades['pnl'].mean() if len(short_trades) > 0 else 0

            stats['long_total_profit'] = long_trades['pnl'].sum() if len(long_trades) > 0 else 0
            stats['short_total_profit'] = short_trades['pnl'].sum() if len(short_trades) > 0 else 0

        # Monthly analysis
        if 'entry_date' in trades_df.columns:
            trades_df['entry_month'] = pd.to_datetime(trades_df['entry_date']).dt.to_period('M')
            monthly_stats = trades_df.groupby('entry_month')['pnl'].agg(['sum', 'count', 'mean'])

            if not monthly_stats.empty:
                stats['best_month'] = str(monthly_stats['sum'].idxmax())
                stats['best_month_profit'] = float(monthly_stats['sum'].max())
                stats['worst_month'] = str(monthly_stats['sum'].idxmin())
                stats['worst_month_profit'] = float(monthly_stats['sum'].min())
                stats['avg_trades_per_month'] = float(monthly_stats['count'].mean())

        # Analyze exit reasons
        if 'exit_signal' in trades_df.columns:
            exit_counts = trades_df['exit_signal'].value_counts()
            for exit_type, count in exit_counts.items():
                stats[f'exit_{exit_type}_count'] = int(count)
                exit_pnl = trades_df[trades_df['exit_signal'] == exit_type]['pnl'].mean()
                stats[f'exit_{exit_type}_avg_pnl'] = float(exit_pnl)

        # Analyze consecutive win/loss patterns
        if 'pnl' in trades_df.columns:
            trades_df['win'] = trades_df['pnl'] > 0

            # Create a column that identifies when the win/loss status changes
            trades_df['status_change'] = trades_df['win'].ne(trades_df['win'].shift())

            # Identify streaks by cumulative sum of status changes
            trades_df['streak_id'] = trades_df['status_change'].cumsum()

            # Count streak lengths
            streak_lengths = trades_df.groupby(['streak_id', 'win']).size()

            # Winning streaks
            winning_streaks = [length for (streak_id, is_win), length in streak_lengths.items() if is_win]
            if winning_streaks:
                stats['avg_winning_streak'] = float(sum(winning_streaks) / len(winning_streaks))

            # Losing streaks
            losing_streaks = [length for (streak_id, is_win), length in streak_lengths.items() if not is_win]
            if losing_streaks:
                stats['avg_losing_streak'] = float(sum(losing_streaks) / len(losing_streaks))

        # Profit factor by trade type
        if 'position' in trades_df.columns and 'pnl' in trades_df.columns:
            # For long trades
            long_wins = float(long_trades[long_trades['pnl'] > 0]['pnl'].sum() if not long_trades.empty else 0)
            long_losses = float(abs(long_trades[long_trades['pnl'] <= 0]['pnl'].sum()) if not long_trades.empty else 0)
            stats['long_profit_factor'] = float(long_wins / long_losses if long_losses != 0 else float('inf'))

            # For short trades
            short_wins = float(short_trades[short_trades['pnl'] > 0]['pnl'].sum() if not short_trades.empty else 0)
            short_losses = float(abs(short_trades[short_trades['pnl'] <= 0]['pnl'].sum()) if not short_trades.empty else 0)
            stats['short_profit_factor'] = float(short_wins / short_losses if short_losses != 0 else float('inf'))

        # Time in market analysis
        if 'duration' in trades_df.columns:
            total_days = (self.data.index[-1] - self.data.index[0]).days
            total_trade_days = float(trades_df['duration'].sum())
            stats['time_in_market_pct'] = float((total_trade_days / total_days) * 100 if total_days > 0 else 0)

        # Print the statistics
        print("\nDetailed Trade Statistics:")
        for key, value in stats.items():
            if 'duration' in key:
                print(f"{key.replace('_', ' ').title()}: {value:.1f} days")
            elif 'percentage' in key or 'rate' in key or 'pct' in key:
                print(f"{key.replace('_', ' ').title()}: {value:.2f}%")
            elif 'profit' in key or 'loss' in key or 'pnl' in key:
                print(f"{key.replace('_', ' ').title()}: ${value:.2f}")
            elif isinstance(value, float):
                print(f"{key.replace('_', ' ').title()}: {value:.2f}")
            elif isinstance(value, int):
                print(f"{key.replace('_', ' ').title()}: {value}")
            else:
                print(f"{key.replace('_', ' ').title()}: {value}")

        return stats

In [ ]:
# Cell 5: Download Stock Data
# -------------------------
def download_stock_data(ticker="SPY", start="2008-01-01", end="2021-12-31"):
    """
    Downloads data for a single ticker from Yahoo Finance,
    then flattens multi-index columns using droplevel.
    The final DataFrame columns will be [Open, High, Low, Close, Volume].

    Parameters:
    -----------
    ticker : str
        The ticker symbol to download
    start : str
        Start date in 'YYYY-MM-DD' format
    end : str
        End date in 'YYYY-MM-DD' format

    Returns:
    --------
    pd.DataFrame
        DataFrame with OHLCV data
    """
    print(f"Downloading data for {ticker} from {start} to {end}...")

    # Download data from Yahoo Finance with group_by='ticker'
    df = yf.download(ticker, start=start, end=end, group_by='ticker')

    # Handle multi-index columns
    df.columns = df.columns.droplevel(0)

    # Make sure the index name is 'Date'
    df.index.name = 'Date'

    print(f"Downloaded {len(df)} bars for {ticker}")
    return df


ticker = 'SPY'
start_date = '2008-01-01'
end_date = '2021-12-31'

# Download the data
data = download_stock_data(ticker, start_date, end_date)

# Display the first few rows of the dataframe
print("\nSample of downloaded data:")
print(data.head())

# Display data info
print("\nDataframe information:")
print(data.info())





In [ ]:

# Cell 6: Add Technical Indicators
# -----------------------------
def add_indicators(df):
    """
    Calculate and add technical indicators to the dataframe

    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame with OHLCV data

    Returns:
    --------
    pd.DataFrame
        DataFrame with added indicators
    """
    print("Adding technical indicators...")

    # Create a copy of the dataframe
    data = df.copy()

    # Simple Moving Averages
    data['SMA20'] = ta.sma(data['Close'], length=20)
    data['SMA50'] = ta.sma(data['Close'], length=50)
    data['SMA200'] = ta.sma(data['Close'], length=200)

    # Exponential Moving Averages
    data['EMA20'] = ta.ema(data['Close'], length=20)
    data['EMA50'] = ta.ema(data['Close'], length=50)
    data['EMA30'] = ta.ema(data['Close'], length=30)
    data['EMA40'] = ta.ema(data['Close'], length=40)

    # Bollinger Bands (20,2)
    bbands = ta.bbands(data['Close'], length=20, std=2)
    data['BB_Upper'] = bbands['BBU_20_2.0']
    data['BB_Middle'] = bbands['BBM_20_2.0']
    data['BB_Lower'] = bbands['BBL_20_2.0']

    # RSI (14)
    data['RSI'] = ta.rsi(data['Close'], length=14)

    # MACD
    macd = ta.macd(data['Close'])
    data['MACD'] = macd['MACD_12_26_9']
    data['MACD_Signal'] = macd['MACDs_12_26_9']
    data['MACD_Hist'] = macd['MACDh_12_26_9']

    # ATR - Average True Range
    data['ATR'] = ta.atr(data['High'], data['Low'], data['Close'], length=14)
    ##supertrend
    supert=ta.supertrend(data['High'], data['Low'], data['Close'])
    data["supertrend"]=supert["SUPERT_7_3.0"]

    # Stochastic Oscillator
    stoch = ta.stoch(data['High'], data['Low'], data['Close'])
    data['STOCH_K'] = stoch['STOCHk_14_3_3']
    data['STOCH_D'] = stoch['STOCHd_14_3_3']

    data['HAMMER'] = detect_hammer(data)
    bullish_engulfing, bearish_engulfing = detect_engulfing(data)
    data['BULLISH_ENGULFING'] = bullish_engulfing
    data['BEARISH_ENGULFING'] = bearish_engulfing

    # Drop NaN values created by indicators that need lookback periods
    # Alternatively, you can keep them by commenting out this line

    data = data.dropna()

    print(f"Added {len(data.columns) - 5} indicators. Data now has {len(data)} rows.")
    return data

# Add indicators to our data
data_with_indicators = add_indicators(data)

# Display the first few rows of the dataframe with indicators
print("\nSample of data with indicators:")
print(data_with_indicators.head())

# Display the list of available columns
print("\nAvailable columns for strategy development:")
for col in data_with_indicators.columns:
    print(f"- {col}")

In [ ]:

# long only strategy : if close[0] below lower bollinger band  buy and exit when low[0]> exponential  moving average 20

In [ ]:
def time_based_exit_strategy(data):
    signals = {}

    # Get price and indicator series
    bars_since_entry = data['bars_since_entry']

    shooting_star=data['SHOOTING_STAR']
    # Enter on bearish engulfing
    if shooting_star[0]:
        signals['enter_short'] = True


    # Exit after exactly 5 bars
    if bars_since_entry == 5:
        signals['exit_short'] = True


    return signals

# To run this debug strategy:

# Create a backtest engine
backtest = BacktestEngine(data_with_indicators, ticker, initial_capital=100000)

# Run the backtest with debug output to see if bars_since_entry is working properly
results = backtest.run_backtest(time_based_exit_strategy, trade_amount=10000, debug=False)

# Analyze the results
backtest.plot_results()
stats = backtest.analyze_trade_statistics()

# Check trade durations to see if 5-bar exits are working
print("\nTrade duration analysis:")
durations = [trade['duration'] for trade in backtest.trades if 'duration' in trade]
if durations:
    print(f"Average duration: {sum(durations)/len(durations):.1f} days")
    print(f"Duration counts: {pd.Series(durations).value_counts().sort_index()}")


In [ ]:
# # Strategy Development EXAMPLE
def hammer_reversal_strategy(data):
    """
    Hammer candlestick reversal strategy with additional context requirements

    Entry conditions:
    1. Yesterday's close was below the MA30 (downtrend confirmation)
    2. A hammer pattern formed yesterday
    3. Current price is below MA20 (still in overall downtrend)
    4. Today's close confirms the reversal by closing above yesterday's high

    Exit conditions:
    1. Price closes above MA20 (trend change)

    Parameters:
    -----------
    data : dict
        Dictionary with price and indicator wrappers

    Returns:
    --------
    dict
        Dictionary with trading signals
    """
    signals = {}

    # Get price and indicator series with EasyLanguage-style indexing
    close = data['close']
    high = data['high']
    ma20 = data['SMA20']
    ma50= data['SMA50']

    hammer = data['HAMMER']

    # ENTRY CONDITIONS
    # Check all conditions for a potential long entry:

    # 1. Yesterday's close was below the MA30 (confirming downtrend)
   # downtrend_confirmed = close[1] < ma30[1]

    # 2. A hammer pattern formed yesterday
    hammer_yesterday = hammer[1]

    # 3. Current price is still below MA20 (we're still in the overall downtrend)
    still_in_downtrend = close[1] < ma20[1]

    # 4. Today's close is above yesterday's high (price confirmation)
    price_confirmation = close[0] > close[1]

    # Combine all conditions for entry
    if  hammer_yesterday and still_in_downtrend and price_confirmation:
        signals['enter_long'] = True

        # Optional: Print debugging information when running with debug=True
        # print(f"ENTRY SIGNAL: Hammer pattern detected with confirmation")
        # print(f"  Yesterday's close ${close[1]:.2f} < MA30 ${ma30[1]:.2f}")
        # print(f"  Hammer pattern detected yesterday")
        # print(f"  Current close ${close[0]:.2f} < MA20 ${ma20[0]:.2f}")
        # print(f"  Current close ${close[0]:.2f} > Yesterday's high ${high[1]:.2f}")

    # EXIT CONDITION
    # Exit when price closes above  a moving average (indicating potential trend change)
    if close[0] > ma50[0]:
        signals['exit_long'] = True

    return signals

# Example of how to use this strategy in the backtest engine:
backtest = BacktestEngine(data_with_indicators, ticker, initial_capital=100000)
results = backtest.run_backtest(hammer_reversal_strategy, trade_amount=10000, stop_loss_usd=2000)
backtest.plot_results()
backtest.analyze_trade_statistics()

In [ ]:

# Research Workspace
# ----------------------
# This is where you  can develop your own strategies
def my_strategy(data):
    """
    My custom trading strategy

    Parameters:
    -----------
    data : dict
        Dictionary with price and indicator wrappers

    Returns:
    --------
    dict
        Dictionary with trading signals
    """
    signals = {}

    # Get price and indicator series with EasyLanguage-style indexing
    close = data['close']
    rsi = data['RSI']
    bb_lower = data['BB_Lower']
    bb_upper = data['BB_Upper']
    high = data['high']
    sma20 = data['SMA20']
    # EXAMPLE: Bollinger Band Strategy with EasyLanguage-style syntax
    # Enter long when price closes below lower Bollinger Band
    if close[0] < bb_lower[0]:
        signals['enter_long'] = True

    # Exit long when price closes above previous high plus 0.2%
    if close[0] > sma20[0]:  # 0.2% above previous high
        signals['exit_long'] = True

    # Enter short when RSI is above 70
    if rsi[0] > 70:
        signals['enter_short'] = False

    # Exit short when RSI falls below 30
    if rsi[0] < 30:
        signals['exit_short'] = False

    # Using the helper functions for lowest/highest
    # Example: Buy if current close is lower than the lowest close of the last 5 bars
    if close[0] < close.lowest(5):
        signals['enter_long'] = True

    return signals
backtest = BacktestEngine(data_with_indicators, ticker, initial_capital=100000)
results = backtest.run_backtest( my_strategy, trade_amount=10000, stop_loss_usd=200, debug=False)
backtest.plot_results()
backtest.analyze_trade_statistics()

In [ ]:
data.head()